# Orquestrador Comparativo — Flow Shop Scheduling

Notebook que reúne todos os solvers dos alunos e os compara usando
as mesmas instancias, seeds e tbm com as mesmas metricas.

**Solvers:** ACO (aluno_1) | Neuro-BOA (aluno_3) | BRKGA (aluno_4) | PBIL-Fuzzy (aluno_2)
**Engine:** `FlowShopEngine` oficial compartilhada
**Modo:** NPFS (Non-Permutation Flow Shop)
**Objetivo principal:** Makespan (Tardiness também é calculado)

## 1. Setup — Instalação e Imports

Rode esta célula para instalar as dependências e carregar todos os módulos.

In [ ]:
# ============================================================
# Instalação das dependências necessárias
# ============================================================
!pip install -q pymoo scikit-fuzzy optuna numpy matplotlib pandas scipy

# PyTorch (CPU basta para o Neuro-BOA)
import torch
if not torch.cuda.is_available():
    !pip install -q torch --index-url https://download.pytorch.org/whp/cpu

print("Dependências instaladas.")

In [ ]:
#imports
import os, sys, time, copy, random, itertools
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('default')

# Engine passada
sys.path.insert(0, "/content/FlowShopScheduling")
from src.flowshop_engine import FlowShopEngine

# Solver adapter
from src.solvers.adapter import (
    rodar_experimento,
    listar_solvers_disponiveis,
    SolverResult,
)

print("Imports concluídos.")
print(f"Solvers disponíveis: {listar_solvers_disponiveis()}")

## 2. Configuração dos Experimentos

In [ ]:
# CONFIGURACAO GERAL -> EDITAR AQUI ANTES DE RODAR EM!
# --- Diretórios ---
DATA_DIR = Path("/content/FlowShopScheduling/data/Small")
RESULTADOS_DIR = Path("/content/resultados_comparacao")
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

# --- Instâncias a testar ---
# Filtro por padrão de nome: formato I_{m}_{n}_{d}_{id}.txt
# Exemplo: I_2_16_5_1.txt = 2 máquinas, 16 jobs, d=5, id=1
INSTANCIAS_CONFIG = [
    {"rotulo": "4x2",  "padrao": "I_2_4_",   "desc": "4 jobs, 2 máq. — smoke test"},
    {"rotulo": "16x2", "padrao": "I_2_16_2_", "desc": "16 jobs, 2 máq. — instância média"},
    {"rotulo": "16x5", "padrao": "I_2_16_5_", "desc": "16 jobs, 5 máq. — instância alvo (aluno_4)"},
    {"rotulo": "16x4", "padrao": "I_4_16_5_", "desc": "16 jobs, 4 máq. — instância grande"},
]

# --- Solvers e variantes ---
SOLVERS_A_TESTAR = [
    "ACO",
    "BRKGA_fixo_default",
    "BRKGA_fuzzy",
    "Neuro-BOA_adaptativo",
    "PBIL-Fuzzy",
]

# --- Parâmetros de execução ---
N_SEEDS = 5                     # sementes por solver × instância
SEED_INICIO = 100                # seeds: range(INICIO, INICIO + N_SEEDS)
N_GERACOES = 100                # gerações
OBJECTIVE = "Makespan"

print(f"Serão testados {len(SOLVERS_A_TESTAR)} solvers em {len(INSTANCIAS_CONFIG)} instâncias")
print(f"com {N_SEEDS} seeds cada = {len(SOLVERS_A_TESTAR) * len(INSTANCIAS_CONFIG) * N_SEEDS} execuções")

## 3. Seleção de Instâncias

Escolhe automaticamente os arquivos `.txt` que correspondem aos padrões configurados

In [ ]:
def selecionar_instancias(data_dir: Path, configs: List[Dict]) -> Dict[str, Path]:
    if not data_dir.exists():
        raise FileNotFoundError(f"Diretório de instâncias não encontrado: {data_dir}")

    todos = sorted(data_dir.glob("*.txt"))
    if not todos:
        raise FileNotFoundError(f"Nenhum arquivo .txt encontrado em {data_dir}")

    escolhidas = {}
    for cfg in configs:
        candidatos = [f for f in todos if cfg["padrao"] in f.name]
        if not candidatos:
            print(f"Instância '{cfg['rotulo']}' ({cfg['padrao']}*) não encontrada — pulando")
            continue
        # Pega a primeira (id=1) se disponível
        escolhidas[cfg["rotulo"]] = candidatos[0]
        print(f"{cfg['rotulo']}: {candidatos[0].name}  ({cfg['desc']})")

    if not escolhidas:
        raise RuntimeError("Nenhuma instância foi encontrada! Verifique DATA_DIR.")

    return escolhidas


instancias = selecionar_instancias(DATA_DIR, INSTANCIAS_CONFIG)

## 4. Loop de Experimentos

Para cada instância × solver × seed, executa o solver e coleta o resultado.

In [ ]:
def montar_params(solver_name: str, n_gen: int) -> dict:
    """Retorna os hiperparâmetros adequados para cada solver."""
    params = {}
    if solver_name == "ACO":
        params = {
            "n_ants": 30,
            "n_iterations": n_gen,
            "alpha": 0.5,
            "beta": 3.0,
            "rho": 0.3,
            "Q": 200.0,
            "mode": "NPFS",
        }
    elif solver_name.startswith("BRKGA"):
        params = {
            "n_gen": n_gen,
            "n_elites": 20,
            "n_offsprings": 70,
            "n_mutants": 10,
            "bias": 0.7,
        }
    elif solver_name.startswith("Neuro-BOA"):
        params = {
            "generations": n_gen,
            "population_size": 40,
            "elite_frac": 0.15,
            "neuro_hidden": 128,
            "neuro_epochs_per_gen": 2,
            "probe_per_generator": 4,
        }
    elif solver_name == "PBIL-Fuzzy":
        params = {
            "max_geracoes": n_gen,
            "n_pop": 80,
            "sigma_amostragem": 0.10,
            "pct_elite": 0.08,
        }
    return params


# Lista para acumular resultados
resultados_brutos = []  # lista de dicts
resultados_detalhados = {}  # (rotulo, solver, seed) -> SolverResult

print("=" * 70)
print("INICIANDO EXPERIMENTOS")
print("=" * 70)

total_exec = len(instancias) * len(SOLVERS_A_TESTAR) * N_SEEDS
exec_atual = 0

for rotulo_inst, inst_path in instancias.items():
    print(f"\n--- Instância: {rotulo_inst} ({inst_path.name}) ---")

    # Carrega a instancia na engine
    n_jobs, n_mach, proc_times, due_dates = FlowShopEngine.carregar_instancia_txt(
        str(inst_path)
    )
    engine = FlowShopEngine(n_jobs, n_mach, proc_times, due_dates)
    print(f"  Jobs: {n_jobs}, Maquinas: {n_mach}")

    for solver_name in SOLVERS_A_TESTAR:
        print(f"Solver: {solver_name}")

        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            exec_atual += 1
            params = montar_params(solver_name, N_GERACOES)

            if exec_atual % 5 == 0:
                print(f"    [{exec_atual}/{total_exec}] seed {seed}...")

            try:
                result = rodar_experimento(
                    solver_name=solver_name,
                    engine=engine,
                    seed=seed,
                    objective=OBJECTIVE,
                    params=params,
                    verbose=False,
                )

                resultados_brutos.append({
                    "instancia_rotulo": rotulo_inst,
                    "instancia_arquivo": inst_path.name,
                    "n_jobs": n_jobs,
                    "n_machines": n_mach,
                    "solver": result.solver_name,
                    "versao": result.version,
                    "seed": seed,
                    "makespan": result.best_cost,
                    "tardiness": result.tardiness,
                    "gap_pct": round(result.gap_percent(), 2),
                    "tempo_s": round(result.time_seconds, 2),
                    "n_avaliacoes": result.n_evaluations,
                    "lower_bound": result.lower_bound,
                })

                resultados_detalhados[(rotulo_inst, solver_name, seed)] = result

            except Exception as e:
                print(f"    ERRO em {solver_name} seed {seed}: {e}")
                resultados_brutos.append({
                    "instancia_rotulo": rotulo_inst,
                    "instancia_arquivo": inst_path.name,
                    "n_jobs": n_jobs,
                    "n_machines": n_mach,
                    "solver": solver_name,
                    "versao": "",
                    "seed": seed,
                    "makespan": float("nan"),
                    "tardiness": float("nan"),
                    "gap_pct": float("nan"),
                    "tempo_s": float("nan"),
                    "n_avaliacoes": 0,
                    "lower_bound": float("nan"),
                })

print("\nExperimentos concluídos!")

In [ ]:
# Converte para DataFrame e salva CSV
df_resultados = pd.DataFrame(resultados_brutos)
csv_path = RESULTADOS_DIR / "resultados_comparativos.csv"
df_resultados.to_csv(csv_path, index=False)
print(f"Resultados salvos em: {csv_path}")
print(f"Total de linhas: {len(df_resultados)}")
print(f"Colunas: {list(df_resultados.columns)}")

## 5. Tabela Comparativa

Para cada instância, mostra: makespan médio, desvio, melhor, tardiness médio, tempo médio, gap.

In [ ]:
def gerar_tabela_comparativa(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa por instância e solver, calcula estatísticas.
    Retorna DataFrame formatado.
    """
    agrupado = df.groupby(["instancia_rotulo", "solver"], dropna=False)

    linhas = []
    for (inst, solver), grupo in agrupado:
        makespans = grupo["makespan"].dropna()
        if len(makespans) == 0:
            continue
        linhas.append({
            "Instância": inst,
            "Solver": solver,
            "Makespan μ": f"{makespans.mean():.1f}",
            "Makespan σ": f"{makespans.std():.1f}",
            "Melhor": f"{makespans.min():.1f}",
            "Pior": f"{makespans.max():.1f}",
            "Tardiness μ": f"{grupo['tardiness'].mean():.1f}",
            "Tempo (s)": f"{grupo['tempo_s'].mean():.2f}",
            "Gap (%) μ": f"{grupo['gap_pct'].mean():.2f}",
        })

    return pd.DataFrame(linhas)


tabela = gerar_tabela_comparativa(df_resultados)
print("=" * 110)
print("TABELA COMPARATIVA — Makespan por Instância × Solver")
print("=" * 110)
print(tabela.to_string(index=False))

## 6. Gráficos

### 6.1 Boxplot — distribuição do makespan por solver (instância alvo 16×5)

In [ ]:
def plotar_boxplot(df: pd.DataFrame, instancia_alvo: str = "16x5"):
    """Boxplot comparando a distribuição de makespan entre solvers."""
    df_alvo = df[df["instancia_rotulo"] == instancia_alvo].dropna(subset=["makespan"])

    if df_alvo.empty:
        print(f"Sem dados para instância {instancia_alvo}")
        return

    solvers_unicos = df_alvo["solver"].unique()
    dados = [df_alvo[df_alvo["solver"] == s]["makespan"].values for s in solvers_unicos]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(dados, labels=solvers_unicos, patch_artist=True)

    cores = ["#1E2761", "#2C7A57", "#E8871E", "#8C2F39", "#9AA6C3"]
    for patch, cor in zip(bp["boxes"], cores[:len(solvers_unicos)]):
        patch.set_facecolor(cor)
        patch.set_alpha(0.6)

    ax.set_xlabel("Solver")
    ax.set_ylabel("Makespan")
    ax.set_title(f"Distribuição do Makespan — Instância {instancia_alvo} ({N_SEEDS} seeds)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"boxplot_{instancia_alvo}.png"), dpi=150)
    plt.show()


plotar_boxplot(df_resultados, "16x5")

### 6.2 Convergência — makespan × geração para uma seed (instância alvo)

In [ ]:
def plotar_convergencia(resultados_detalhados, instancia_alvo="16x5", seed_plot=None):
    """
    Plota a curva de convergência de cada solver para uma seed específica.
    """
    # Pega a seed do meio se não especificada
    if seed_plot is None:
        seeds_disponiveis = set(
            s for (i, _, s) in resultados_detalhados if i == instancia_alvo
        )
        seed_plot = sorted(seeds_disponiveis)[len(seeds_disponiveis) // 2]

    fig, ax = plt.subplots(figsize=(11, 5))

    cores = {
        "ACO": "#1E2761",
        "BRKGA": "#2C7A57",
        "Neuro-BOA": "#E8871E",
        "PBIL-Fuzzy": "#8C2F39",
    }

    for (inst, solver_name, seed), result in sorted(resultados_detalhados.items()):
        if inst != instancia_alvo or seed != seed_plot:
            continue
        history = result.history
        if not history:
            continue
        label = f"{result.solver_name} ({result.version})"
        cor = cores.get(result.solver_name, "gray")
        ax.plot(history, label=label, color=cor, linewidth=1.8)

    ax.set_xlabel("Iteração / Geração")
    ax.set_ylabel("Melhor Makespan")
    ax.set_title(f"Convergência — Instância {instancia_alvo} (seed {seed_plot})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"convergencia_{instancia_alvo}_seed{seed_plot}.png"), dpi=150)
    plt.show()


plotar_convergencia(resultados_detalhados, "16x5")

### 6.3 Gap para Lower Bound — comparativo

In [ ]:
def plotar_gap(df: pd.DataFrame, instancia_alvo="16x5"):
    """Barra do gap médio para lower bound por solver."""
    df_alvo = df[df["instancia_rotulo"] == instancia_alvo].dropna(subset=["gap_pct"])
    if df_alvo.empty:
        return

    medias = df_alvo.groupby("solver")["gap_pct"].agg(["mean", "std"])

    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = range(len(medias))
    bars = ax.bar(x, medias["mean"], yerr=medias["std"],
                  capsize=5, color=["#1E2761", "#2C7A57", "#E8871E", "#8C2F39", "#9AA6C3"][:len(medias)])
    ax.set_xticks(x)
    ax.set_xticklabels(medias.index, rotation=15)
    ax.set_ylabel("Gap (%) para Lower Bound")
    ax.set_title(f"Gap para Lower Bound — Instância {instancia_alvo}")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"gap_{instancia_alvo}.png"), dpi=150)
    plt.show()


plotar_gap(df_resultados, "16x5")

### 6.4 Heatmap — tempo de execução

In [ ]:
def plotar_heatmap_tempo(df: pd.DataFrame):
    """Heatmap do tempo médio por instância × solver."""
    pivot = df.pivot_table(
        values="tempo_s", index="instancia_rotulo", columns="solver",
        aggfunc="mean"
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=20)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Tempo Médio de Execução (s)")

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                        color="white" if val > pivot.values[~np.isnan(pivot.values)].mean() else "black")

    plt.colorbar(im, ax=ax, label="Tempo (s)")
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / "heatmap_tempo.png"), dpi=150)
    plt.show()


plotar_heatmap_tempo(df_resultados)

## 7. Resumo Final

Exibe um resumo geral dos experimentos.

In [ ]:
print("=" * 60)
print("RESUMO FINAL DOS EXPERIMENTOS")
print("=" * 60)

# Melhor solver por instância (menor makespan médio)
melhores_por_instancia = df_resultados.loc[
    df_resultados.groupby("instancia_rotulo")["makespan"].idxmin()
]
print("\nMelhor solver (menor makespan médio) por instância:")
for _, row in melhores_por_instancia.iterrows():
    print(f"  {row['instancia_rotulo']}: {row['solver']} ({row['makespan']:.1f})")

# Solver mais rápido (menor tempo médio)
mais_rapido = df_resultados.groupby("solver")["tempo_s"].mean().idxmin()
tempo_rapido = df_resultados.groupby("solver")["tempo_s"].mean().min()
print(f"\nSolver mais rápido: {mais_rapido} ({tempo_rapido:.2f}s médios)")

# Menor gap
menor_gap = df_resultados.groupby("solver")["gap_pct"].mean().idxmin()
gap_valor = df_resultados.groupby("solver")["gap_pct"].mean().min()
print(f"\nMenor gap para LB: {menor_gap} ({gap_valor:.2f}% médio)")

# Arquivos gerados
print("\nArquivos gerados:")
for f in sorted(RESULTADOS_DIR.glob("*")):
    print(f"  {f.name}")